# Human 배치 snapshot — 보드 정보 기준선
## 1. 왜 / 무엇
지금까지는 정답 목표를 알려주고 조작을 예측했습니다. 이번에는 **보드·조각 종류에서 Human 배치 목표를 예측**합니다.

| 조건 | 조각 종류 | boardBefore 점유 |
|---|---|---|
| piece_only | O | 200차원 0-mask |
| board_piece | O | O |

두 모델 모두 207 → ReLU64 → Train 물리 배치 class의 softmax입니다. **별도 Smoke 학습은 없습니다.**
작은 내부 검사는 코드 보호용이며, 실제 학습은3절 Full에서만 실행합니다.
매 epoch마다 같은 기준의 Train/Validation NLL과 Top-1을 기록합니다. Validation은 관찰용이며 조기 종료나 모델 선택에 쓰지 않습니다.
**저장된 기존 출력은 epoch 계측 추가 전 run1 결과입니다. 새 곡선은 새 경로로 재실행해야 생기며, 과거 값을 추정해 채우지 않습니다.**

[사전 명세](../../../ai-harness/ml-experiments/experiments/human-placement-snapshot-baseline.md)
이것은 **snapshot 관계 탐색**이지 합법/도달 가능한 배치 정책이 아닙니다. 기록 시점·현재40행 호환성은 미검증입니다.


## 2. 입력·정답을 눈으로 구분하기
`boardBefore`를200개의 점유값(null=0, 블록=1)으로 펴고 조각 종류7차원을 붙입니다.

- 입력: 보드 점유 + **activePiece.pieceType만**
- 정답: Human이 선택한 네 셀의 physicalClassKey
- 입력 금지: activePiece의 최종 row/col/rotation/cells, selectedPlacement, oracle 목표, 미래 행동/결과/승패
- 같은 셀을 차지하는 rotation/pivot alias는 같은 정답입니다.

Train에서 본 배치 class만 출력합니다. Validation의 새 class를 몰래 어휘에 넣지 않습니다.
새 class는 Top-1/Top-3/MRR에서 실패(0)로 남기고 coverage를 따로 보고합니다.
출력 목록은 **관측된 Train label 목록**이며 보드별 합법 후보 목록이 아닙니다.


In [ ]:
import copy
import hashlib
import json
import os
import platform
import tempfile
import warnings
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-snapshot-baseline"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_snapshot_baseline.ipynb"
BASE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_goal_conditioned_prefix.ipynb"
BASE_DEFS_SHA = "12f07e2a25b35fdaedd9c3c6ed805e9b6fbb6f5b3fd236569d0363b654b051d2"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(BASE_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == BASE_DEFS_SHA
base = {"__name__": "placement_snapshot_shared"}
for cell in selected:
    if cell["id"] in {"shared-definitions", "feature-definitions", "model-definitions"}:
        exec(compile(cell["source"], str(BASE_NOTEBOOK), "exec"), base)
for name in ("require", "finite", "integer", "json_bytes", "sha256_file", "definition_sha",
             "repo_input", "physical_key", "fit_head", "predict_probabilities",
             "checked_paths", "publish_report", "reject_constant", "finite_float", "threadpool_limits"):
    globals()[name] = base[name]
base["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
PIECES = base["PIECES"]
VARIANTS = ("piece_only", "board_piece")
MANIFEST_SHA = base["EXPECTED_MANIFEST_SHA"]
EXPECTED = {"train": (6440, 6417, 149, base["EXPECTED_FILES"]["train"][-1]),
            "validation": (1519, 1516, 31, base["EXPECTED_FILES"]["validation"][-1])}


In [ ]:
def input_features(board, piece_type):
    """boardBefore 점유와 조각 종류만 받는다. 배치 좌표/정답을 받는 인자가 없다."""
    require(piece_type in PIECES and isinstance(board, list) and len(board) == 20
            and all(isinstance(row, list) and len(row) == 10 and all(
                c is None or isinstance(c, str) and bool(c.strip()) for c in row) for row in board),
            "BOARD_OR_PIECE_INVALID")
    return np.array([float(c is not None) for row in board for c in row]
                    + [float(piece_type == p) for p in PIECES], dtype=float)

def placement_row(segment, split):
    lineage, placement = segment["lineage"], segment["placement"]
    require(lineage["split"] == split and segment["support"]["policy"] == "offline-use-v1"
            and placement["offlineOnly"] is True and placement["inferenceEligible"] is False,
            "SNAPSHOT_CONTRACT_INVALID")
    snapshot, target = placement["inputSnapshot"], placement["target"]
    piece_type = snapshot["activePiece"]["pieceType"]
    features = input_features(snapshot["boardBefore"], piece_type)
    cells = sorted(copy.deepcopy(target["selectedCells"]), key=lambda c: (c["row"], c["col"]))
    key = physical_key({"pieceType": piece_type, "rotation": 0, "row": 0, "col": 0, "cells": cells}, selected=True)
    require(key == target["physicalClassKey"], "TARGET_CLASS_MISMATCH")
    return {"segmentId": segment["segmentId"], "split": split, "gameId": lineage["gameId"],
            "sourceLine": lineage["sourceLine"], "pieceType": piece_type,
            "x": features, "targetKey": key, "targetCells": cells}

def train_vocabulary(train):
    require(train and all(r["split"] == "train" for r in train), "VOCABULARY_TRAIN_ONLY")
    keys = sorted({r["targetKey"] for r in train})
    require(len(keys) >= 2, "INSUFFICIENT_TRAIN_CLASSES")
    return keys

def masked_features(scaled, variant):
    require(variant in VARIANTS, "VARIANT_INVALID")
    result = scaled.copy()
    if variant == "piece_only":
        result[:, :200] = 0
    return result

def load_snapshots():
    directory = repo_input(os.environ.get("HUMAN_PLACEMENT_DATASET",
        "data/datasets/human-placement-action-offline-v1-20261102-run1"))
    paths = {s: directory / (s + ".jsonl") for s in EXPECTED}
    paths["manifest"] = directory / "manifest.json"
    before = {p: sha256_file(p) for p in paths.values()}
    require(before[paths["manifest"]] == MANIFEST_SHA, "MANIFEST_SHA_MISMATCH")
    manifest = json.loads(paths["manifest"].read_text(encoding="utf-8"), parse_constant=reject_constant)
    require(manifest["status"] == "offline_dataset_constructed" and manifest["policy"] == "offline-use-v1"
            and manifest["testPayloadRead"] is False, "DATASET_CONTRACT_INVALID")
    data = {}
    for split, (raw_count, used_count, games, digest) in EXPECTED.items():
        require(before[paths[split]] == digest
                and manifest["files"][split + ".jsonl"]["sha256"] == digest, "DATASET_SHA_MISMATCH")
        with paths[split].open("rb") as handle:
            segments = [json.loads(line, parse_constant=reject_constant, parse_float=finite_float) for line in handle]
        require(len(segments) == raw_count, "SOURCE_ROW_COUNT_CHANGED")
        rows = [placement_row(s, split) for s in segments
                if s["eligibility"]["placement_snapshot_eligible"]["eligible"]]
        rows.sort(key=lambda r: (r["sourceLine"], r["segmentId"]))
        require(len(rows) == used_count and len({r["gameId"] for r in rows}) == games
                and len({r["segmentId"] for r in rows}) == len(rows), "ELIGIBLE_COHORT_CHANGED")
        data[split] = rows
    reserved = set(manifest["provenance"]["reservedTestGameIds"])
    groups = {s: {r["gameId"] for r in rows} for s, rows in data.items()}
    require(len(reserved) == 33 and not groups["train"] & groups["validation"]
            and not (groups["train"] | groups["validation"]) & reserved, "GAME_SPLIT_LEAKAGE")
    require(all(sha256_file(p) == h for p, h in before.items()), "DATASET_CHANGED")
    return data, before


In [ ]:
def rank_probabilities(rows, probability, vocabulary):
    """같은 조각의 Train class만 순위화한다. 합법성/도달 가능성은 판정하지 않는다."""
    require(probability.shape == (len(rows), len(vocabulary)) and np.isfinite(probability).all()
            and (probability >= 0).all(), "PROBABILITY_INVALID")
    key_to_id = {key: i for i, key in enumerate(vocabulary)}
    piece_ids = {piece: [i for i, key in enumerate(vocabulary) if key.split("|")[0] == piece] for piece in PIECES}
    scored = []
    for row, p in zip(rows, probability):
        ids = piece_ids[row["pieceType"]]
        known = row["targetKey"] in key_to_id
        if not ids:
            scored.append({"gameId": row["gameId"], "pieceType": row["pieceType"], "rank": None,
                           "knownTarget": known, "knownTargetNll": None, "logClipped": False,
                           "predictedKey": None, "selectionAvailable": False, "vocabularySizeForPiece": 0})
            continue
        values = p[ids].copy()
        mass = float(values.sum())
        require(mass > 0 and finite(mass), "PIECE_PROBABILITY_MASS_ZERO")
        values /= mass
        order = np.argsort(-values, kind="stable")
        ranked_ids = [ids[int(i)] for i in order]
        rank = ranked_ids.index(key_to_id[row["targetKey"]]) + 1 if known else None
        true_probability = float(values[ids.index(key_to_id[row["targetKey"]])]) if known else None
        scored.append({"gameId": row["gameId"], "pieceType": row["pieceType"], "rank": rank,
                       "knownTarget": known,
                       "knownTargetNll": -float(np.log(max(true_probability, 1e-15))) if known else None,
                       "logClipped": known and true_probability < 1e-15,
                       "predictedKey": vocabulary[ranked_ids[0]], "selectionAvailable": True,
                       "vocabularySizeForPiece": len(ids)})
    return scored

def metric_summary(scored):
    require(scored, "EMPTY_EVALUATION")
    known = [r for r in scored if r["knownTarget"]]
    supported = [r for r in known if r["knownTargetNll"] is not None]
    n = len(scored)
    return {"segments": n, "top1": sum(r["rank"] == 1 for r in scored) / n,
            "top3": sum(r["rank"] is not None and r["rank"] <= 3 for r in scored) / n,
            "mrr": sum(1 / r["rank"] for r in scored if r["rank"] is not None) / n,
            "knownTargetCoverage": len(known) / n, "unknownTargetSegments": n - len(known),
            "selectionUnavailableSegments": sum(not r["selectionAvailable"] for r in scored),
            "knownTargetNll": float(np.mean([r["knownTargetNll"] for r in supported])) if supported else None,
            "knownNllDenominator": len(supported),
            "knownTargetMeanRank": float(np.mean([r["rank"] for r in known])) if known else None,
            "fullTargetNll": float(np.mean([r["knownTargetNll"] for r in supported])) if len(supported) == n else None,
            "fullTargetNllUnsupportedReason": None if len(supported) == n else "TARGET_OUTSIDE_TRAIN_VOCABULARY",
            "probabilityLogClipCount": sum(r["logClipped"] for r in scored)}

def evaluate_rankings(scored):
    per_game = [{"gameId": g, **metric_summary([r for r in scored if r["gameId"] == g])}
                for g in sorted({r["gameId"] for r in scored})]
    per_piece = [{"pieceType": p, **metric_summary([r for r in scored if r["pieceType"] == p])}
                 for p in PIECES if any(r["pieceType"] == p for r in scored)]
    macro = {k: float(np.mean([g[k] for g in per_game])) for k in ("top1", "top3", "mrr", "knownTargetCoverage")}
    return {"gameMacro": macro, "segmentMicro": metric_summary(scored), "perGame": per_game, "perPiece": per_piece}


In [ ]:
def fit_with_validation(train_x, y, validation_x, train_rows, validation_rows, vocabulary, epochs):
    """매 epoch 종료 후 같은 NLL/Top-1을 측정한다. Validation으로 학습/모델 선택하지 않는다."""
    require(integer(epochs) and epochs >= 1, "INVALID_EPOCH_BUDGET")
    # 기존 초기화/optimizer를 재사용하고 같은 모델의 partial_fit을 이어간다.
    model, losses = fit_head(train_x, y, np.arange(len(vocabulary)), 1729, 1)
    history = []
    for epoch in range(epochs):
        if epoch:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                model.partial_fit(train_x, y)
            require(not caught, "UNEXPECTED_FIT_WARNING: " + "; ".join(
                f"{w.category.__name__}: {w.message}" for w in caught))
            require(np.isfinite(model.loss_), "NONFINITE_TRAIN_LOSS")
            losses.append(float(model.loss_))
        train_ranked = rank_probabilities(train_rows,
            predict_probabilities(model, train_x, len(vocabulary)), vocabulary)
        validation_ranked = rank_probabilities(validation_rows,
            predict_probabilities(model, validation_x, len(vocabulary)), vocabulary)
        history.append({"epoch": epoch + 1, "train": metric_summary(train_ranked),
                        "validation": metric_summary(validation_ranked)})
        print(f"epoch {epoch + 1}/{epochs}: train NLL={history[-1]['train']['knownTargetNll']}, "
              f"val NLL={history[-1]['validation']['knownTargetNll']} (known targets only)")
    require(all(np.isfinite(a).all() for a in (*model.coefs_, *model.intercepts_)), "NONFINITE_MODEL")
    return model, losses, history, validation_ranked

def run_full():
    require((platform.python_version(), np.__version__, base["sklearn"].__version__)
            == ("3.12.13", "2.5.1", "1.9.0"), "PINNED_TRAINING_ENVIRONMENT_REQUIRED")
    before_code = definition_sha(SOURCE_NOTEBOOK)
    data, watched = load_snapshots()
    vocabulary = train_vocabulary(data["train"])
    require(len(vocabulary) == 2133, "TRAIN_VOCABULARY_COUNT_CHANGED")
    ids = {key: i for i, key in enumerate(vocabulary)}
    train_x = np.stack([r["x"] for r in data["train"]])
    y = np.array([ids[r["targetKey"]] for r in data["train"]], dtype=int)
    scaler = base["StandardScaler"]().fit(train_x)
    scaled_train = scaler.transform(train_x)
    scaled_validation = scaler.transform(np.stack([r["x"] for r in data["validation"]]))
    results, losses, rankings, histories = {}, {}, {}, {}
    print("Full eligible cohort: Train6417 / Validation1516 segments; vocabulary2133; fitting2 models ×30epochs.")
    with threadpool_limits(limits=1):
        for variant in VARIANTS:
            print(variant)
            model, loss, history, ranked = fit_with_validation(
                masked_features(scaled_train, variant), y, masked_features(scaled_validation, variant),
                data["train"], data["validation"], vocabulary, 30)
            rankings[variant] = ranked
            results[variant] = evaluate_rankings(rankings[variant])
            losses[variant] = loss
            histories[variant] = history
            require(history[-1]["validation"] == results[variant]["segmentMicro"]
                    and all(h["train"]["knownNllDenominator"] == len(y)
                            and h["validation"]["knownNllDenominator"] == 1364
                            and h["validation"]["unknownTargetSegments"] == 152 for h in history),
                    "EPOCH_METRIC_DENOMINATOR_CHANGED")
    a, b = (results[v]["gameMacro"] for v in VARIANTS)
    delta = {k: b[k] - a[k] for k in a}
    status = ("board_snapshot_signal_observed" if delta["top1"] > 1e-12 and delta["mrr"] >= -1e-12 else
              "board_snapshot_tradeoff_requires_review" if delta["top1"] > 1e-12 else "no_clear_board_snapshot_gain")
    per_game_delta = [{"gameId": a["gameId"], **{k: b[k] - a[k] for k in delta}}
                      for a, b in zip(results[VARIANTS[0]]["perGame"], results[VARIANTS[1]]["perGame"])]
    cells_by_key = {r["targetKey"]: r["targetCells"] for r in data["train"]}
    examples = [{"sourceLine": r["sourceLine"], "pieceType": r["pieceType"],
                 "boardOccupancy": r["x"][:200].reshape(20, 10).astype(int).tolist(),
                 "targetCells": r["targetCells"], "predictedCells": cells_by_key.get(pred["predictedKey"]),
                 "targetKnown": pred["knownTarget"], "rank": pred["rank"]}
                for r, pred in list(zip(data["validation"], rankings["board_piece"]))[:3]]
    require(all(sha256_file(p) == h for p, h in watched.items())
            and definition_sha(SOURCE_NOTEBOOK) == before_code
            and definition_sha(BASE_NOTEBOOK) == BASE_DEFS_SHA, "INPUT_OR_CODE_CHANGED")
    return {"reportSchemaVersion": 2, "experimentId": EXPERIMENT_ID, "runMode": "full",
            "executionPolicy": "full-only", "fullCohortCovered": True, "policyDecisionEligible": True,
            "canonicalResultEligible": False, "productionIntegration": False,
            "wholePolicyEvidence": False, "inferenceEligible": False, "legalityChecked": False,
            "reachableCandidatesGenerated": False, "snapshotObservation": "unverified",
            "modelFitCount": 2, "smokeFitCount": 0, "analysisStatus": status,
            "decisionScope": "closed_vocabulary_snapshot_association_only",
            "results": results, "boardMinusPieceGameMacro": delta, "perGameDelta": per_game_delta,
            "trainingLoss": losses, "epochMetrics": histories, "boardExamples": examples,
            "epochMetricContract": {"aggregation": "segment_micro", "measuredAfterEpoch": True,
                                    "nll": "same_piece_known_target_nll_without_l2",
                                    "accuracy": "top1_all_targets_oov_is_failure",
                                    "probabilityFloor": 1e-15},
            "training": {"segments": len(y), "games": 149, "epochs": 30, "featureDimensions": 207,
                         "hiddenSizes": [64], "outputClasses": len(vocabulary), "seed": 1729,
                         "batchSize": 128, "optimizer": "adam", "learningRate": 0.001, "alpha": 0.0001,
                         "convergenceAssessed": False, "fitWarnings": 0, "validationUsedForFit": False,
                         "validationUsedForMonitoring": True, "validationUsedForModelSelection": False,
                         "scalerFitSegments": int(scaler.n_samples_seen_),
                         "scalerMean": scaler.mean_.tolist(), "scalerScale": scaler.scale_.tolist()},
            "vocabulary": {"fitSplit": "train", "classCount": len(vocabulary),
                           "sha256": hashlib.sha256(json_bytes(vocabulary)).hexdigest(),
                           "classesPerPiece": dict(sorted(Counter(k.split("|")[0] for k in vocabulary).items())),
                           "classSupportHistogram": dict(sorted(Counter(Counter(r["targetKey"] for r in data["train"]).values()).items()))},
            "sourceDefinitionSha256": before_code, "baseDefinitionSha256": BASE_DEFS_SHA,
            "sourceUnchanged": True, "testFileRead": False, "rawFileRead": False,
            "provenance": {"manifestSha256": MANIFEST_SHA, "datasetFiles": {s: EXPECTED[s][-1] for s in EXPECTED}},
            "environment": {"python": platform.python_version(), "numpy": np.__version__,
                            "sklearn": base["sklearn"].__version__, "blasThreads": 1}}

def report_path(name):
    require(isinstance(name, str) and bool(name), "NEW_FULL_REPORT_PATH_REQUIRED")
    result = Path(name)
    if not result.is_absolute():
        result = ROOT / result
    allowed = ROOT / "data/ml-experiments" / EXPERIMENT_ID
    require(".." not in result.parts and not any(p.is_symlink() for p in (result, *result.parents)),
            "OUTPUT_ALIAS_FORBIDDEN")
    require(result.parent == allowed or result.parent == Path("/private/tmp")
            or str(result.parent).startswith("/private/tmp/"), "OUTPUT_LOCATION_NOT_ALLOWED")
    require(not result.exists() and not result.is_symlink() and result.suffix == ".json",
            "REPORT_ALREADY_EXISTS_OR_INVALID")
    result.parent.mkdir(parents=True, exist_ok=True)
    return checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(result), "full")[1]


In [ ]:
def internal_checks():
    """누수·순위·경로·epoch 계측을 검사한다. 신경망 대신 가짜 모델을 사용한다."""
    cells = [{"row": r, "col": c} for r in (18, 19) for c in (0, 1)]
    key = physical_key({"pieceType": "square", "rotation": 0, "row": 18, "col": 0, "cells": cells}, selected=True)
    segment = {"segmentId": "check", "lineage": {"split": "train", "gameId": "A", "sourceLine": 1},
               "support": {"policy": "offline-use-v1"},
               "placement": {"offlineOnly": True, "inferenceEligible": False,
                             "inputSnapshot": {"boardBefore": [[None] * 10 for _ in range(20)],
                                               "activePiece": {"pieceType": "square", "row": 18, "col": 0,
                                                               "rotation": 0, "cells": copy.deepcopy(cells)}},
                             "target": {"physicalClassKey": key, "selectedCells": copy.deepcopy(cells)}}}
    original = json_bytes(segment)
    row = placement_row(segment, "train")
    changed = copy.deepcopy(segment)
    changed["placement"]["inputSnapshot"]["activePiece"].update(row=-99, col=999, rotation=123, cells=[])
    changed["outcome"] = "win"
    changed["boardAfter"] = "forbidden-future-value"
    assert np.array_equal(row["x"], placement_row(changed, "train")["x"])
    reversed_segment = copy.deepcopy(segment)
    reversed_segment["placement"]["target"]["selectedCells"].reverse()
    assert placement_row(reversed_segment, "train")["targetKey"] == key
    other = copy.deepcopy(segment)
    for cell in other["placement"]["target"]["selectedCells"]:
        cell["row"] -= 2
    other_key = physical_key({"pieceType": "square", "rotation": 0, "row": 0, "col": 0,
                              "cells": other["placement"]["target"]["selectedCells"]}, selected=True)
    other["placement"]["target"]["physicalClassKey"] = other_key
    other_row = placement_row(other, "train")
    assert np.array_equal(row["x"], other_row["x"]) and other_row["targetKey"] != row["targetKey"]
    vocabulary = train_vocabulary([row, other_row])
    require(len(vocabulary) == 2, "CHECK_VOCABULARY")
    known = {**row, "gameId": "A", "targetKey": vocabulary[0]}
    unseen = {**row, "gameId": "A", "targetKey": "square|unseen"}
    unseen_piece = {**row, "gameId": "B", "pieceType": "bar", "targetKey": "bar|unseen"}
    ranked = rank_probabilities([known, unseen, unseen_piece], np.full((3, 2), 0.5), vocabulary)
    assert ranked[0]["rank"] == 1 and ranked[0]["predictedKey"] == vocabulary[0]
    assert ranked[1]["rank"] is None and ranked[2]["selectionAvailable"] is False
    evaluation = evaluate_rankings(ranked)
    assert np.isclose(evaluation["segmentMicro"]["top1"], 1 / 3)
    assert evaluation["gameMacro"]["top1"] == 0.25
    assert evaluation["segmentMicro"]["unknownTargetSegments"] == 2
    assert evaluation["segmentMicro"]["fullTargetNll"] is None
    assert np.isclose(evaluation["segmentMicro"]["knownTargetNll"], np.log(2))
    try:
        train_vocabulary([{**row, "split": "validation"}, other_row])
    except ValueError:
        pass
    else:
        raise AssertionError("VALIDATION_VOCABULARY_FIT_ALLOWED")
    matrix = np.arange(414, dtype=float).reshape(2, 207)
    masked = masked_features(matrix, "piece_only")
    assert not masked[:, :200].any() and np.array_equal(masked[:, 200:], matrix[:, 200:])
    assert json_bytes(segment) == original
    labels = np.array([0, 1])
    train_rows = [known, {**known, "targetKey": vocabulary[1]}]
    validation_rows = [known, unseen, unseen_piece]
    class FakeModel:
        classes_ = np.arange(2)
        coefs_, intercepts_ = [np.ones((1, 1))], [np.zeros(1)]
        def __init__(self):
            self.steps = 0
        def partial_fit(self, x, y):
            assert np.array_equal(x, matrix) and np.array_equal(y, labels)
            self.steps += 1
            self.loss_ = np.float64(1 / self.steps)
            return self
        def predict_proba(self, x):
            p = 1 - 0.5 ** (self.steps + 1)
            return np.tile([p, 1 - p], (len(x), 1))
    def fake_fit(x, y, classes, seed, epochs):
        assert epochs == 1 and seed == 1729 and np.array_equal(classes, np.arange(2))
        model = FakeModel().partial_fit(x, y)
        return model, [model.loss_]
    original_fit = globals()["fit_head"]
    try:
        globals()["fit_head"] = fake_fit
        model, loss, history, last = fit_with_validation(
            matrix, labels, np.zeros((3, 207)), train_rows, validation_rows, vocabulary, 2)
        assert model.steps == 2 and loss == [1.0, 0.5] and [h["epoch"] for h in history] == [1, 2]
        assert history[-1]["validation"] == metric_summary(last)
        for h, p in zip(history, (0.75, 0.875)):
            assert np.isclose(h["train"]["knownTargetNll"], -(np.log(p) + np.log(1 - p)) / 2)
            assert np.isclose(h["validation"]["knownTargetNll"], -np.log(p))
            assert h["train"]["top1"] == 0.5 and h["validation"]["top1"] == 1 / 3
            assert h["validation"]["knownNllDenominator"] == 1
            assert h["validation"]["unknownTargetSegments"] == 2 and h["validation"]["fullTargetNll"] is None
    finally:
        globals()["fit_head"] = original_fit
    with tempfile.TemporaryDirectory(dir="/private/tmp") as directory:
        path = report_path(str(Path(directory) / "check.json"))
        publish_report({"internalCheckOnly": True}, path)
        (path.parent / "alias.json").symlink_to(path)
        (path.parent / "dangling.json").symlink_to(path.parent / "missing.json")
        os.link(path, path.parent / "hardlink.json")
        for name in ("check.json", "alias.json", "dangling.json", "hardlink.json"):
            try:
                report_path(str(path.parent / name))
            except ValueError:
                pass
            else:
                raise AssertionError("OUTPUT_OVERWRITE_ALLOWED")
    return "PASS: no target/final-pose features / alias / Train-only vocabulary / OOV denominator / ties / piece mask / epoch NLL and accuracy / Train-only updates / output safety"


In [ ]:
def show_report(report):
    require(report.get("reportSchemaVersion") == 2 and "epochMetrics" in report,
            "EPOCH_METRICS_MISSING_RESTART_KERNEL_AND_RUN_TO_NEW_REPORT_PATH")
    print(report["analysisStatus"], "| snapshot association only; NOT a legal/reachable gameplay policy")
    display(pd.DataFrame({v: report["results"][v]["gameMacro"] for v in VARIANTS}).T)
    display(pd.DataFrame({v: report["results"][v]["segmentMicro"] for v in VARIANTS}).T)
    display(pd.DataFrame(report["results"]["board_piece"]["perPiece"]))
    display(pd.DataFrame(report["perGameDelta"]))
    print("Epoch NLL: same-piece known targets only, without L2. Top-1: all targets, OOV is failure.")
    history_table = pd.DataFrame([{"variant": v, "epoch": h["epoch"],
        "trainNll": h["train"]["knownTargetNll"], "valNll": h["validation"]["knownTargetNll"],
        "trainTop1": h["train"]["top1"], "valTop1": h["validation"]["top1"],
        "trainN": h["train"]["segments"], "valN": h["validation"]["segments"],
        "valNllN": h["validation"]["knownNllDenominator"],
        "valOov": h["validation"]["unknownTargetSegments"]}
        for v in VARIANTS for h in report["epochMetrics"][v]])
    with pd.option_context("display.max_rows", len(history_table)):
        display(history_table)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for col, v in enumerate(VARIANTS):
        history = report["epochMetrics"][v]
        for split, style in (("train", "-"), ("validation", "--")):
            nll_n, total = history[-1][split]["knownNllDenominator"], history[-1][split]["segments"]
            epochs = [h["epoch"] for h in history]
            axes[0, col].plot(epochs, [h[split]["knownTargetNll"] for h in history], style,
                              label=f"{split} known n={nll_n}/{total}")
            axes[1, col].plot(epochs, [h[split]["top1"] for h in history], style,
                              label=f"{split} all n={total}")
        axes[0, col].set(xlabel="epoch", ylabel="Known-target NLL (no L2)", title=v)
        axes[1, col].set(xlabel="epoch", ylabel="Top-1 accuracy (all targets)", ylim=(0, 1), title=v)
        axes[0, col].legend(); axes[1, col].legend()
    fig.suptitle("After each epoch — monitoring only, no early stopping / model selection")
    plt.tight_layout(); plt.show(); plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    pd.DataFrame({v: {k: report["results"][v]["gameMacro"][k] for k in ("top1", "top3", "mrr")}
                  for v in VARIANTS}).plot.bar(ax=axes[0], rot=0)
    axes[0].set(ylim=(0, 1), title="Game macro: all Validation targets, OOV is failure")
    pieces = pd.DataFrame(report["results"]["board_piece"]["perPiece"]).set_index("pieceType")
    pieces["knownTargetCoverage"].plot.bar(ax=axes[1], rot=25)
    axes[1].set(ylim=(0, 1), ylabel="Train-label coverage", title="Closed-vocabulary limitation")
    plt.tight_layout(); plt.show(); plt.close(fig)
    fig, axes = plt.subplots(1, len(report["boardExamples"]), figsize=(12, 5), squeeze=False)
    for ax, example in zip(axes[0], report["boardExamples"]):
        ax.imshow(example["boardOccupancy"], cmap="Greys", vmin=0, vmax=1)
        target = example["targetCells"]
        ax.scatter([c["col"] for c in target], [c["row"] for c in target],
                   marker="s", s=100, facecolors="none", edgecolors="green", label="Human target")
        if example["predictedCells"]:
            pred = example["predictedCells"]
            ax.scatter([c["col"] for c in pred], [c["row"] for c in pred],
                       marker="x", s=70, c="tab:orange", label="Predicted Train class")
        ax.set(title=f"line {example['sourceLine']} / known={example['targetKnown']}",
               xlabel="col", ylabel="row", ylim=(19.5, -0.5), xlim=(-0.5, 9.5))
        ax.legend(fontsize=7)
    fig.suptitle("First 3 Validation snapshots — legality / reachability NOT checked")
    plt.tight_layout(); plt.show(); plt.close(fig)


## 3. 내부 검사 / Full 설정·실행
기본 모두 실행은 짧은 코드 검사만 합니다. 신경망을 작은 표본으로 학습하지 않습니다.

실제 Full은 아래 설정을 바꾸고 실행·결과 셀을 실행합니다.
```python
RUN_FULL = False
FULL_REPORT = "data/ml-experiments/human-placement-snapshot-baseline/full-report-run2.json"
```

두 모델을 동일30epoch로 학습합니다. Train의2133개 class를 예측하므로 조작6-class 모델보다 출력층이 큽니다.
출력은 report뿐이며 weights/production artifact를 만들지 않습니다. 기존 report는 새 이름 없이 재사용할 수 없습니다.
사용자의 기존 설정/출력은 보존했습니다. **외부 코드 수정 후에는 열린 노트북을 디스크에서 다시 불러오고 커널을 재시작한 뒤 위에서부터 실행**하세요. 저장되지 않은 개인 수정은 먼저 별도로 보관하고, 오래된 편집기 내용을 덮어쓰지 마세요. FULL_REPORT는 기존 report와 겹치지 않는 run2 같은 새 경로를 사용합니다.
각 epoch의 Train/Validation NLL은 같은 조각 class 내 재정규화·L2 제외 기준입니다. Validation NLL은 known1364개만, Top-1은 OOV152개를 포함한1516개 전체입니다.


In [ ]:
RUN_FULL = False
FULL_REPORT = "data/ml-experiments/human-placement-snapshot-baseline/full-report-run2.json"

In [ ]:
print(internal_checks())
full_report = None
if RUN_FULL:
    output = report_path(FULL_REPORT)
    full_report = run_full()
    publish_report(full_report, output)
    print("Saved:", str(output), "| SHA-256:", sha256_file(output))
else:
    print("Full disabled. Only internal checks ran; no neural fit or performance evaluation.")


## 4. 결과만 다시 읽기
이미 Full을 실행했다면 RUN_FULL을 켜지 말고 아래 REPORT_TO_READ에 저장된 JSON 경로를 넣습니다.
첫 표는 게임별 평균, 두 번째 표는 전체 구간 기준입니다. coverage와 OOV를 먼저 확인한 뒤 Top-1/Top-3/MRR를 읽으세요.
epoch 표/그림은 구간 기준 NLL·Top-1입니다. Train NLL은 감소하는데 Validation NLL이 올라가면 과적합 징후입니다. trainingLoss는 별도 CE+L2 최적화 진단으로 보존하며 이 곡선과 섞지 않습니다.


In [ ]:
REPORT_TO_READ = None
if REPORT_TO_READ:
    path = Path(REPORT_TO_READ)
    if not path.is_absolute():
        path = ROOT / path
    loaded = json.loads(path.read_text(encoding="utf-8"))
    require(loaded["reportSchemaVersion"] == 2 and loaded["experimentId"] == EXPERIMENT_ID and loaded["runMode"] == "full"
            and loaded["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK)
            and loaded["baseDefinitionSha256"] == BASE_DEFS_SHA, "STALE_OR_DIFFERENT_REPORT")
    full_report = loaded


In [ ]:
if full_report is not None:
    show_report(full_report)


## 5. 해석·한계·다음 gate
- 보드 조건이 좋아지면 현재 snapshot의 점유 정보가 조각별 습관만으로는 설명되지 않는 선택을 설명한다는 탐색 근거입니다.
- activePiece 최종 위치/정답을 직접 입력에서 제거했지만, boardBefore가 행동 직전 관측이라는 보장은 없습니다.
- 출력은 **Train에서 본 class**만입니다. 처음 보는 Validation 배치는 전체 지표에서 실패로 남습니다. known-only NLL을 전체 성능으로 읽지 마세요.
- epoch마다 Validation을 관찰하지만 학습/조기 종료/선택에는 쓰지 않습니다. 두 split의 NLL 정의는 같아도 대상 분포는 다르며, Validation의 OOV는 NLL에서 제외하고 별도 보고합니다.
- 조각 종류 mask는 사용하지만 충돌·지지·배치 경로·게임오버를 검사하지 않습니다. 그림의 예측 셀이 보드와 겹쳐도 조용히 보정하지 않습니다.
- 기존 V4처럼 게임별 합법 후보를 평가한 점수와 직접 비교하지 않습니다. 이 분류기를 바로 controller의 목표 생성기로 연결하지 않습니다.
- 희소2133-class·작은 데이터·표현/예산 제약이 있습니다. 낮은 점수가 사람 모방 불가능이나 새 수집 필요를 뜻하지 않습니다.
- 다음은 사용자 Full 결과의 coverage·게임/조각별 편차 해석입니다. 새 수집·Hold·replay·예측 목표 rollout·runtime·추가 실험을 자동 시작하지 않습니다.
